In [ ]:
import os
# Configuración de numpy
import numpy as np

# Configuración de pandas
import pandas as pd

# Configuración de polars
import polars as pl

# Configuración de matplotlib
import matplotlib.pyplot as plt

# Configuración de seaborn
import seaborn as sns

# Configuración de psycopg2
import psycopg2

# Configuración de unicode
from unidecode import unidecode

# Configuración de re
import re

# Configuración de nbformat
import nbformat

# Configuración de json
import json

# Configuración de datetime
from datetime import datetime

# Configuración de openpyxl
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

In [ ]:
# Definir la consulta
consulta = '''
select distinct cluster_index, cluster_name, main_chain, country
from esquema_analitico.dim_locales dm
where country in ('cl', 'mx')
and app_estado_tienda in ('Activa', 'Inactiva')
'''

# Crear la conexión a la base de datos
conn = psycopg2.connect(
    dbname=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    host=os.environ["DB_HOST"],
    port=os.environ.get("DB_PORT", "5432")
)

# Ejecutar la consulta y leer los datos en un DataFrame de Polars
with conn.cursor() as cursor:
    cursor.execute(consulta)
    data = cursor.fetchall()
    column_names = [desc[0] for desc in cursor.description]

# Crear el DataFrame de Polars
df = pl.DataFrame(data, schema=column_names)

# Mostrar el DataFrame
df

In [ ]:
# Definir la consulta para leer los datos de la tabla esquema_analitico.ctrl_categorias_keywords
consulta_keywords = '''
SELECT *
FROM esquema_analitico.ctrl_categorias_keywords
'''

# Ejecutar la consulta y leer los datos en un DataFrame de Polars
with conn.cursor() as cursor:
    cursor.execute(consulta_keywords)
    data = cursor.fetchall()
    column_names = [desc[0] for desc in cursor.description]

# Crear el DataFrame de Polars
df2 = pl.DataFrame(data, schema=column_names)

# Mostrar el DataFrame
df2

In [ ]:
# Función personalizada para normalizar y separar caracteres no alfabéticos y eliminar números
def normalize_text(text):
    if isinstance(text, str):
        text = unidecode(text.lower().replace('"', '').strip())
        text = re.sub(r'^\+', '', text)  # Remover el signo '+' al inicio del texto
        text = re.sub(r'\+$', '', text)  # Remover el signo '+' al final del texto
        text = re.sub(r'\s+', ' ', text)  # Reemplazar múltiples espacios con uno solo
        text = re.sub(r'\d+', '', text)  # Eliminar números
        text = re.sub(r"([^a-zA-Z\s'])", r' \1 ', text)  # Separar caracteres no alfabéticos y comillas dobles, exceptuando comillas simples
        text = re.sub(r'\s+', ' ', text)  # Reemplazar múltiples espacios con uno solo después de la separación
        text = text.strip()  # Eliminar espacios al principio y al final
        return text
    return text  # En caso de que el texto no sea un string, devolverlo sin cambios

# Realizar el tratamiento de datos
df = df.with_columns([
    pl.col('cluster_name').map_elements(normalize_text, return_dtype=pl.Utf8).alias('cluster_name'),
    pl.col('main_chain').map_elements(normalize_text, return_dtype=pl.Utf8).alias('main_chain')
])

# Mostrar el DataFrame actualizado
df

In [ ]:
# Tratamiento de datos en df2
df2 = df2.with_columns([
    pl.col('categoria').map_elements(normalize_text, return_dtype=pl.Utf8).alias('categoria'),
    pl.col('keyword').map_elements(normalize_text, return_dtype=pl.Utf8).alias('keyword'),
    pl.col('tipo_keyword').map_elements(normalize_text, return_dtype=pl.Utf8).alias('tipo_keyword'),
    pl.col('tipo_clasificacion').map_elements(normalize_text, return_dtype=pl.Utf8).alias('tipo_clasificacion')
])

# Mostrar el DataFrame actualizado
df2

In [ ]:
# Crear la nueva columna 'main_chain_grouped' basado en la condición utilizando 'contains'
df = df.with_columns([
    pl.when(pl.col('main_chain').str.contains('comercio'))
    .then(pl.lit('comercio'))
    .otherwise(pl.lit('no comercio'))
    .alias('main_chain_grouped')
])

# Contar las ocurrencias de cada valor en 'main_chain_grouped' para 'cl'
counts_cl = df.filter(pl.col('country') == 'cl').group_by('main_chain_grouped').agg(
    pl.col('main_chain_grouped').count().alias('count_cl')
).sort('main_chain_grouped')

# Contar las ocurrencias de cada valor en 'main_chain_grouped' para 'mx'
counts_mx = df.filter(pl.col('country') == 'mx').group_by('main_chain_grouped').agg(
    pl.col('main_chain_grouped').count().alias('count_mx')
).sort('main_chain_grouped')

# Mostrar los resultados
print("Conteos para 'cl':")
print(counts_cl)

print("\nConteos para 'mx':")
print(counts_mx)

In [ ]:
# Convertir los resultados a listas para usarlos en Matplotlib
labels_cl = counts_cl['main_chain_grouped'].to_list()
sizes_cl = counts_cl['count_cl'].to_list()

labels_mx = counts_mx['main_chain_grouped'].to_list()
sizes_mx = counts_mx['count_mx'].to_list()

# Configurar el tema de Seaborn
sns.set_theme(style='whitegrid')

# Crear una paleta de colores de negro a azul
unique_cl = len(labels_cl)
unique_mx = len(labels_mx)
max_unique = max(unique_cl, unique_mx)
palette = sns.dark_palette("blue", reverse=False, as_cmap=False, n_colors=max_unique)

# Crear la figura y los ejes para los gráficos en una grilla de 1x2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Crear el gráfico de sectores para 'cl'
palette_cl = sns.color_palette(palette[:unique_cl])
wedges_cl, texts_cl, autotexts_cl = ax1.pie(sizes_cl, autopct='%1.1f%%', startangle=90, colors=palette_cl, textprops={'color': "w"})
ax1.set_title("Chile", fontsize=22)

# Añadir una leyenda debajo del gráfico de 'cl'
ax1.legend(wedges_cl, labels_cl, title="Categorías", loc="lower center", ncol=2)

# Crear el gráfico de sectores para 'mx'
palette_mx = sns.color_palette(palette[:unique_mx])
wedges_mx, texts_mx, autotexts_mx = ax2.pie(sizes_mx, autopct='%1.1f%%', startangle=90, colors=palette_mx, textprops={'color': "w"})
ax2.set_title("México", fontsize=22)

# Añadir una leyenda debajo del gráfico de 'mx'
ax2.legend(wedges_mx, labels_mx, title="Categorías", loc="lower center", ncol=2)

# Ajustar la disposición para evitar solapamientos
plt.tight_layout()
plt.subplots_adjust(top=0.85)

# Guardar los gráficos como un archivo PNG con fondo transparente
plt.savefig('graficos_sectores_cl_mx.png', transparent=True, bbox_inches='tight')

# Mostrar los gráficos
plt.show()

print("Los gráficos de sectores han sido guardados como 'graficos_sectores_cl_mx.png'")

In [ ]:
# Función para asignar categorías basándose en el identifier y las keywords, incluyendo tipo_clasificacion
def assign_categories(identifier, keywords_df):
    categories = []
    criterio_clasificacion = []
    tipo_clasificacion = 'nombre'

    # Prioridad 1: nombre local (excluyendo 'de origen' y 'otros')
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'nombre local') & 
        (pl.col('categoria') != 'de origen') & 
        (pl.col('categoria') != 'otros')
    ).iter_rows(named=True):
        if row['keyword'] == identifier:
            categories.append(row['categoria'])
            criterio_clasificacion.append('nombre local')
            tipo_clasificacion = row['tipo_clasificacion']
            return categories, criterio_clasificacion, tipo_clasificacion  # Detener si se encuentra una categoría por nombre local

    # Prioridad 2: palabra clave entera, vegano y vegetariano
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave entera') & 
        (pl.col('categoria') == 'vegano & vegetariano')
    ).iter_rows(named=True):
        pattern = r'\b' + re.escape(row['keyword']) + r'\b'
        if re.search(pattern, identifier):
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave entera')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 3: palabra clave fraccion inicio, vegano y vegetariano
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave fraccion inicio') & 
        (pl.col('categoria') == 'vegano & vegetariano')
    ).iter_rows(named=True):
        pattern = r'\b' + re.escape(row['keyword'])
        if re.search(pattern, identifier):
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave fraccion inicio')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 4: palabra clave fraccion azar, vegano y vegetariano
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave fraccion azar') & 
        (pl.col('categoria') == 'vegano & vegetariano')
    ).iter_rows(named=True):
        if row['keyword'] in identifier:
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave fraccion azar')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 5: palabra clave entera, 'pollo' o 'a las brasas'
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave entera') & 
        ((pl.col('categoria') == 'pollo') | (pl.col('categoria') == 'a las brasas'))
    ).iter_rows(named=True):
        pattern = r'\b' + re.escape(row['keyword']) + r'\b'
        if re.search(pattern, identifier):
            # Anular 'pollo' y 'a las brasas' en favor de 'pollo'
            if 'pollo' in identifier and 'a las brasas' in identifier:
                categories = ['pollo']
                criterio_clasificacion = ['palabra clave entera']
                tipo_clasificacion = row['tipo_clasificacion']
                break
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave entera')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 6: palabra clave fraccion inicio, 'pollo' o 'a las brasas'
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave fraccion inicio') & 
        ((pl.col('categoria') == 'pollo') | (pl.col('categoria') == 'a las brasas'))
    ).iter_rows(named=True):
        pattern = r'\b' + re.escape(row['keyword'])
        if re.search(pattern, identifier):
            # Anular 'pollo' y 'a las brasas' en favor de 'pollo'
            if 'pollo' in identifier and 'a las brasas' in identifier:
                categories = ['pollo']
                criterio_clasificacion = ['palabra clave fraccion inicio']
                tipo_clasificacion = row['tipo_clasificacion']
                break
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave fraccion inicio')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 7: palabra clave fraccion azar, 'pollo' o 'a las brasas'
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave fraccion azar') & 
        ((pl.col('categoria') == 'pollo') | (pl.col('categoria') == 'a las brasas'))
    ).iter_rows(named=True):
        if row['keyword'] in identifier:
            # Anular 'pollo' y 'a las brasas' en favor de 'pollo'
            if 'pollo' in identifier and 'a las brasas' in identifier:
                categories = ['pollo']
                criterio_clasificacion = ['palabra clave fraccion azar']
                tipo_clasificacion = row['tipo_clasificacion']
                break
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave fraccion azar')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 8: palabra clave entera, 'amasanderia'
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave entera') & 
        (pl.col('categoria') == 'amasanderia')
    ).iter_rows(named=True):
        pattern = r'\b' + re.escape(row['keyword']) + r'\b'
        if re.search(pattern, identifier):
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave entera')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 9: palabra clave fraccion inicio, 'amasanderia'
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave fraccion inicio') & 
        (pl.col('categoria') == 'amasanderia')
    ).iter_rows(named=True):
        pattern = r'\b' + re.escape(row['keyword'])
        if re.search(pattern, identifier):
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave fraccion inicio')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 10: palabra clave fraccion azar, 'amasanderia'
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave fraccion azar') & 
        (pl.col('categoria') == 'amasanderia')
    ).iter_rows(named=True):
        if row['keyword'] in identifier:
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave fraccion azar')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 11: palabra clave entera (excluyendo 'de origen' y 'otros')
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave entera') & 
        (pl.col('categoria') != 'de origen') & 
        (pl.col('categoria') != 'otros') & 
        (pl.col('categoria') != 'vegano & vegetariano') & 
        (pl.col('categoria') != 'pollo') & 
        (pl.col('categoria') != 'amasanderia')
    ).iter_rows(named=True):
        pattern = r'\b' + re.escape(row['keyword']) + r'\b'
        if re.search(pattern, identifier):
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave entera')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 12: palabra clave fraccion inicio (excluyendo 'de origen' y 'otros')
    for row in keywords_df.filter(
        (pl.col('tipo_keyword').str.contains('palabra clave fraccion inicio')) & 
        (pl.col('categoria') != 'de origen') & 
        (pl.col('categoria') != 'otros') & 
        (pl.col('categoria') != 'vegano & vegetariano') & 
        (pl.col('categoria') != 'pollo') & 
        (pl.col('categoria') != 'amasanderia')
    ).iter_rows(named=True):
        pattern = r'\b' + re.escape(row['keyword'])
        if re.search(pattern, identifier):
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave fraccion inicio')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 13: palabra clave fraccion azar (excluyendo 'de origen' y 'otros')
    for row in keywords_df.filter(
        (pl.col('tipo_keyword') == 'palabra clave fraccion azar') & 
        (pl.col('categoria') != 'de origen') & 
        (pl.col('categoria') != 'otros') & 
        (pl.col('categoria') != 'vegano & vegetariano') & 
        (pl.col('categoria') != 'pollo') & 
        (pl.col('categoria') != 'amasanderia')
    ).iter_rows(named=True):
        if row['keyword'] in identifier:
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append('palabra clave fraccion azar')
                tipo_clasificacion = row['tipo_clasificacion']

    # Prioridad 14: de origen (solo si no se encontró ninguna categoría antes)
    for row in keywords_df.filter(
        (pl.col('categoria') == 'de origen') & 
        (pl.col('categoria') != 'otros')
    ).iter_rows(named=True):
        if row['keyword'] in identifier:
            if row['categoria'] not in categories:
                categories.append(row['categoria'])
                criterio_clasificacion.append(row['tipo_keyword'])
                tipo_clasificacion = row['tipo_clasificacion']

    # Asignar 'otros' solo si no se encontró ninguna otra categoría
    if len(categories) == 0:
        categories.append('otros')
        criterio_clasificacion.append('sin coincidencia')

    # Imprimir si el local entra en más de una categoría
    if len(categories) > 1:
        print(f"El identificador '{identifier}' ha sido asignado a múltiples categorías: {categories}")

    return categories, criterio_clasificacion, tipo_clasificacion

In [ ]:
# Función para reclasificar los registros de "no comercio" clasificados como "otros"
def reclasificar_no_comercio(df, keywords_df):
    # Filtrar los registros que son "no comercio" y están clasificados como "otros"
    no_comercio_otros = df.filter(
        (pl.col('main_chain_grouped') == 'no comercio') & 
        (pl.col('categoria_principal') == 'otros')
    )
    
    # Crear un diccionario para almacenar las nuevas categorías
    new_categories = {}
    
    # Iterar sobre los registros de "no comercio" clasificados como "otros"
    for row in no_comercio_otros.iter_rows(named=True):
        main_chain = row['main_chain']
        cluster_name = row['cluster_name']
        
        # Asignar categorías basadas en cluster_name
        assigned_categories, assigned_criterios, tipo_clasificacion = assign_categories(cluster_name, keywords_df)
        
        # Contar las ocurrencias de cada categoría
        category_counts = {}
        for category in assigned_categories:
            if category in category_counts:
                category_counts[category] += 1
            else:
                category_counts[category] = 1
        
        # Determinar si alguna categoría tiene más del 50%+1 de las ocurrencias
        total_counts = sum(category_counts.values())
        for category, count in category_counts.items():
            if count > total_counts / 2:
                new_categories[main_chain] = (category, assigned_categories, assigned_criterios, tipo_clasificacion)
                break
    
    # Actualizar las categorías en el DataFrame original
    for main_chain, (new_category_principal, new_categories_list, new_criterios_list, tipo_clasificacion) in new_categories.items():
        df = df.with_columns([
            pl.when(pl.col('main_chain') == main_chain)
            .then(pl.lit(new_category_principal))
            .otherwise(pl.col('categoria_principal'))
            .alias('categoria_principal'),
            pl.when(pl.col('main_chain') == main_chain)
            .then(pl.lit(new_categories_list))
            .otherwise(pl.col('categorias'))
            .alias('categorias'),
            pl.when(pl.col('main_chain') == main_chain)
            .then(pl.lit(new_criterios_list))
            .otherwise(pl.col('criterio_clasificacion'))
            .alias('criterio_clasificacion'),
            pl.when(pl.col('main_chain') == main_chain)
            .then(pl.lit(tipo_clasificacion))
            .otherwise(pl.col('tipo_clasificacion'))
            .alias('tipo_clasificacion')
        ])
    
    return df

# Clasificar primero los main_chain únicos para 'No comercio'
unique_main_chains = df.filter(pl.col('main_chain_grouped') == 'no comercio')['main_chain'].unique().to_list()
main_chain_categories = {main_chain: assign_categories(main_chain, df2) for main_chain in unique_main_chains}

# Crear un DataFrame de las categorías asignadas por main_chain único
df_main_chain_categories = pl.DataFrame({
    'main_chain': list(main_chain_categories.keys()),
    'categorias': [x[0] for x in main_chain_categories.values()],
    'criterio_clasificacion': [x[1] for x in main_chain_categories.values()],
    'tipo_clasificacion': [x[2] for x in main_chain_categories.values()]
})

df_main_chain_categories = df_main_chain_categories.with_columns([
    pl.Series('categoria_principal', [x[0] for x in df_main_chain_categories['categorias']])
])

# Unir las categorías asignadas al DataFrame original
df = df.join(df_main_chain_categories, on='main_chain', how='left')

# Clasificar los registros que son 'Comercio' por cluster_name
unique_cluster_names = df.filter(pl.col('main_chain_grouped') == 'comercio')['cluster_name'].unique().to_list()
cluster_name_categories = {cluster_name: assign_categories(cluster_name, df2) for cluster_name in unique_cluster_names}

# Crear un DataFrame de las categorías asignadas por cluster_name
df_cluster_name_categories = pl.DataFrame({
    'cluster_name': list(cluster_name_categories.keys()),
    'categorias': [x[0] for x in cluster_name_categories.values()],
    'criterio_clasificacion': [x[1] for x in cluster_name_categories.values()],
    'tipo_clasificacion': [x[2] for x in cluster_name_categories.values()]
})

df_cluster_name_categories = df_cluster_name_categories.with_columns([
    pl.Series('categoria_principal', [x[0] for x in df_cluster_name_categories['categorias']])
])

# Unir las categorías asignadas al DataFrame original para 'Comercio'
df = df.join(df_cluster_name_categories, on='cluster_name', how='left', suffix='_cluster')

# Manejar los duplicados en las categorías asignadas
df = df.with_columns([
    pl.when(pl.col('categoria_principal_cluster').is_null())
    .then(pl.col('categoria_principal'))
    .otherwise(pl.col('categoria_principal_cluster'))
    .alias('categoria_principal'),
    pl.when(pl.col('categorias_cluster').is_null())
    .then(pl.col('categorias'))
    .otherwise(pl.col('categorias_cluster'))
    .alias('categorias'),
    pl.when(pl.col('criterio_clasificacion_cluster').is_null())
    .then(pl.col('criterio_clasificacion'))
    .otherwise(pl.col('criterio_clasificacion_cluster'))
    .alias('criterio_clasificacion'),
    pl.when(pl.col('tipo_clasificacion_cluster').is_null())
    .then(pl.col('tipo_clasificacion'))
    .otherwise(pl.col('tipo_clasificacion_cluster'))
    .alias('tipo_clasificacion')
])

# Reclasificar los registros de "no comercio" clasificados como "otros"
df = reclasificar_no_comercio(df, df2)

# Verificar si hay registros sin categorías asignadas
sin_categorias = df.filter(pl.col('categorias').map_elements(lambda x: len(x) if x is not None else 0, return_dtype=pl.Int64) == 0)
if sin_categorias.height > 0:
    print("Hay registros sin categorías asignadas:")

# Nueva columna con la cantidad de categorías
df = df.with_columns([
    pl.col('categorias').map_elements(lambda x: len(x) if x is not None else 0, return_dtype=pl.Int64).alias('cantidad_categorias')
])

# Actualizar la columna 'cantidad_categorias' para los registros clasificados como "otros"
df = df.with_columns([
    pl.when(pl.col('categoria_principal') == 'otros')
    .then(pl.lit(0))
    .otherwise(pl.col('cantidad_categorias'))
    .alias('cantidad_categorias')
])

# Seleccionar las columnas requeridas
df = df.select([
    'cluster_index', 'cluster_name', 'main_chain', 'categoria_principal', 'categorias', 'criterio_clasificacion', 'cantidad_categorias', 'tipo_clasificacion', 'country'
])

# Mostrar el DataFrame final
df

In [ ]:
# Filtrar por país
df_cl = df.filter(pl.col('country') == 'cl')
df_mx = df.filter(pl.col('country') == 'mx')

In [ ]:
# Función para preparar los datos para el gráfico de barras
def prepare_data(df, country_name):
    df = df.with_columns(
        pl.when(pl.col('cantidad_categorias') == 0)
        .then(pl.lit("0 categorías"))
        .otherwise(pl.lit("1 o más categorías"))
        .alias('categorias_count')
    )

    # Contar las ocurrencias de cada categoría
    categoria_counts = df.group_by('categorias_count').agg(pl.col('cluster_index').n_unique().alias('count'))

    # Calcular los porcentajes
    categoria_counts = categoria_counts.with_columns(
        (pl.col('count') / pl.col('count').sum() * 100).alias('percentage')
    )

    # Convertir a Pandas para graficar con Seaborn
    categoria_counts_pd = categoria_counts.to_pandas()
    categoria_counts_pd['country'] = country_name

    # Ordenar las filas de Pandas DataFrame para asegurar que "0 categorías" aparezca primero
    categoria_counts_pd['categorias_count'] = pd.Categorical(categoria_counts_pd['categorias_count'], categories=["0 categorías", "1 o más categorías"], ordered=True)
    categoria_counts_pd = categoria_counts_pd.sort_values('categorias_count')

    return categoria_counts_pd

# Preparar los datos para Chile y México
categoria_counts_cl = prepare_data(df_cl, 'Chile')
categoria_counts_mx = prepare_data(df_mx, 'México')

# Configurar el estilo de Seaborn
sns.set_theme(style='whitegrid')

# Crear la paleta de colores
palette = sns.dark_palette("blue", reverse=False, as_cmap=False, n_colors=2)

# Crear la grilla de gráficos con Seaborn
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Crear el gráfico de Chile
sns.barplot(ax=axes[0], x='categorias_count', y='count', hue='categorias_count', data=categoria_counts_cl, dodge=False, palette=palette)
axes[0].set_title('Chile', fontsize=22)
axes[0].set_xlabel('')
axes[0].set_ylabel('Cantidad')
for i, bar in enumerate(axes[0].patches):
    height = bar.get_height()
    percentage = categoria_counts_cl.iloc[i]['percentage']
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f'{height} ({percentage:.2f}%)',
        ha='center',
        va='bottom',
        fontsize=12
    )

# Crear el gráfico de México
sns.barplot(ax=axes[1], x='categorias_count', y='count', hue='categorias_count', data=categoria_counts_mx, dodge=False, palette=palette)
axes[1].set_title('México', fontsize=22)
axes[1].set_xlabel('')
axes[1].set_ylabel('')
for i, bar in enumerate(axes[1].patches):
    height = bar.get_height()
    percentage = categoria_counts_mx.iloc[i]['percentage']
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f'{height} ({percentage:.2f}%)',
        ha='center',
        va='bottom',
        fontsize=12
    )

# Ajustar la disposición para evitar solapamientos
plt.tight_layout()
plt.subplots_adjust(top=0.85)

# Guardar los gráficos como un archivo PNG con fondo transparente
plt.savefig('grafico_barras_categorias_cl_mx.png', transparent=True)

# Guardar los gráficos como un archivo PNG con fondo blanco
plt.savefig('grafico_barras_categorias_cl_mx_blanco.png')

# Mostrar los gráficos
plt.show()

print("El gráfico de barras con la cantidad y porcentaje de categorías por país ha sido guardado como 'grafico_barras_categorias_cl_mx.png'")

In [ ]:
# Función para crear los gráficos por país
def plot_categoria_counts(df, country_code, country_name):
    # Contar las ocurrencias de cada main_chain único por categoría principal
    categoria_counts = df.group_by('categoria_principal').agg(pl.col('cluster_index').n_unique().alias('count'))

    # Calcular los porcentajes
    categoria_counts = categoria_counts.with_columns(
        (pl.col('count') / pl.col('count').sum() * 100).alias('percentage')
    )

    # Convertir a pandas para graficar con Seaborn
    categoria_counts_pd = categoria_counts.sort('percentage', descending=True).to_pandas()
    categoria_counts_pd['country'] = country_name

    return categoria_counts_pd

# Crear los DataFrames por país
categoria_counts_cl = plot_categoria_counts(df_cl, 'cl', 'Chile')
categoria_counts_mx = plot_categoria_counts(df_mx, 'mx', 'México')

# Configurar el estilo de Seaborn y la paleta de colores
sns.set_theme(style='whitegrid')
# Definir una paleta de colores para los registros dinámicamente
palette_cl = sns.dark_palette("blue", reverse=False, n_colors=len(categoria_counts_cl['categoria_principal'].unique()))
palette_mx = sns.dark_palette("blue", reverse=False, n_colors=len(categoria_counts_mx['categoria_principal'].unique()))

# Crear la grilla de gráficos con Seaborn
fig, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=False)

# Crear el gráfico de Chile
sns.barplot(ax=axes[0], x='percentage', y='categoria_principal', hue='categoria_principal', data=categoria_counts_cl, dodge=False, palette=palette_cl, order=categoria_counts_cl.sort_values('percentage', ascending=False)['categoria_principal'])
axes[0].set_title('Chile', fontsize=22)
axes[0].set_xlabel('')
axes[0].set_ylabel('Categoría Principal')
for i, bar in enumerate(axes[0].patches):
    width = bar.get_width()
    count = categoria_counts_cl.iloc[i]['count']
    percentage = categoria_counts_cl.iloc[i]['percentage']
    axes[0].text(width + 0.5, bar.get_y() + bar.get_height() / 2, f'{count} ({percentage:.2f}%)', ha='left', va='center')

# Crear el gráfico de México
sns.barplot(ax=axes[1], x='percentage', y='categoria_principal', hue='categoria_principal', data=categoria_counts_mx, dodge=False, palette=palette_mx, order=categoria_counts_mx.sort_values('percentage', ascending=False)['categoria_principal'])
axes[1].set_title('México', fontsize=22)
axes[1].set_xlabel('Porcentaje')
axes[1].set_ylabel('Categoría Principal')
for i, bar in enumerate(axes[1].patches):
    width = bar.get_width()
    count = categoria_counts_mx.iloc[i]['count']
    percentage = categoria_counts_mx.iloc[i]['percentage']
    axes[1].text(width + 0.5, bar.get_y() + bar.get_height() / 2, f'{count} ({percentage:.2f}%)', ha='left', va='center')

# Ajustar los márgenes y añadir un título global
plt.subplots_adjust(top=0.92, hspace=0.3)
fig.suptitle('Porcentaje de Categorías Principales por País', fontsize=18)

# Guardar los gráficos como un archivo PNG con fondo transparente
plt.savefig('grafico_barras_porcentaje_categorias_pais_vertical.png', transparent=True)

# Guardar los gráficos como un archivo PNG con fondo blanco
plt.savefig('grafico_barras_porcentaje_categorias_pais_vertical_not_transparent.png')

# Mostrar los gráficos
plt.show()

print("El gráfico de barras con el porcentaje de categorías principales por país ha sido guardado como 'grafico_barras_porcentaje_categorias_pais_vertical.png'")

In [ ]:
# Función para contar las categorías y calcular porcentajes
def count_and_percentage(df):
    # Contar las ocurrencias de cada cantidad de categorías por cluster_index único
    cantidad_counts = df.group_by('cantidad_categorias').agg(pl.col('cluster_index').n_unique().alias('counts'))

    # Calcular los porcentajes
    cantidad_counts = cantidad_counts.with_columns(
        (pl.col('counts') / pl.col('counts').sum() * 100).alias('percentage')
    )

    return cantidad_counts.sort('cantidad_categorias').to_pandas()

# Obtener los datos para Chile y México
cantidad_counts_cl = count_and_percentage(df_cl)
cantidad_counts_mx = count_and_percentage(df_mx)

# Configurar el estilo de Seaborn
sns.set_theme(style='whitegrid')

# Crear una paleta de colores de negro a azul
unique_cl = cantidad_counts_cl['cantidad_categorias'].nunique()
unique_mx = cantidad_counts_mx['cantidad_categorias'].nunique()
max_unique = max(unique_cl, unique_mx)
palette = sns.dark_palette("blue", reverse=False, n_colors=max_unique)

# Crear la grilla de gráficos con Seaborn
fig, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=False)

# Crear el gráfico de Chile
palette_cl = sns.color_palette(palette[:unique_cl])
sns.barplot(ax=axes[0], x='cantidad_categorias', y='percentage', hue='cantidad_categorias', data=cantidad_counts_cl, palette=palette_cl, dodge=False, legend=False)
axes[0].set_title('Chile', fontsize=22)
axes[0].set_xlabel('')
axes[0].set_ylabel('Porcentaje')
axes[0].set_xticks(range(len(cantidad_counts_cl['cantidad_categorias'])))
axes[0].set_xticklabels(cantidad_counts_cl['cantidad_categorias'])
for i, bar in enumerate(axes[0].patches):
    height = bar.get_height()
    count = cantidad_counts_cl['counts'][i]
    percentage = cantidad_counts_cl['percentage'][i]
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f'{count} ({percentage:.2f}%)',
        ha='center',
        va='bottom',
        fontsize=12
    )

# Crear el gráfico de México
palette_mx = sns.color_palette(palette[:unique_mx])
sns.barplot(ax=axes[1], x='cantidad_categorias', y='percentage', hue='cantidad_categorias', data=cantidad_counts_mx, palette=palette_mx, dodge=False, legend=False)
axes[1].set_title('México', fontsize=22)
axes[1].set_xlabel('Cantidad de Categorías')
axes[1].set_ylabel('Porcentaje')
axes[1].set_xticks(range(len(cantidad_counts_mx['cantidad_categorias'])))
axes[1].set_xticklabels(cantidad_counts_mx['cantidad_categorias'])
for i, bar in enumerate(axes[1].patches):
    height = bar.get_height()
    count = cantidad_counts_mx['counts'][i]
    percentage = cantidad_counts_mx['percentage'][i]
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f'{count} ({percentage:.2f}%)',
        ha='center',
        va='bottom',
        fontsize=12
    )

# Ajustar los márgenes y añadir un título global
plt.subplots_adjust(top=0.92, hspace=0.3)
fig.suptitle('Porcentaje de cluster_index por Cantidad de Categorías por País', fontsize=18)

# Guardar los gráficos como un archivo PNG con fondo transparente
plt.savefig('grafico_barras_cantidad_categorias_cluster_index_vertical.png', transparent=True)

# Guardar los gráficos como un archivo PNG con fondo blanco
plt.savefig('grafico_barras_cantidad_categorias_cluster_index_vertical_not_transparent.png')

# Mostrar los gráficos
plt.show()

print("El gráfico de barras con la cantidad y porcentaje de cluster_index por cantidad de categorías ha sido guardado como 'grafico_barras_cantidad_categorias_cluster_index_vertical.png'")

In [ ]:
# Función para expandir y contar las ocurrencias de los tipos de keywords
def process_df(df):
    # Expandir la lista de tipos de keywords en filas individuales
    df = df.explode('criterio_clasificacion')

    # Contar las ocurrencias de cada tipo de keyword
    keyword_counts = df.group_by('criterio_clasificacion').agg([
        pl.len().alias('counts')
    ]).sort('counts', descending=True)

    # Calcular los porcentajes
    keyword_counts = keyword_counts.with_columns([
        (pl.col('counts') / pl.col('counts').sum() * 100).alias('percentage')
    ])

    # Convertir a Pandas para graficar con Seaborn
    return keyword_counts.to_pandas()

# Procesar los DataFrames de Chile y México
keyword_counts_cl = process_df(df_cl)
keyword_counts_mx = process_df(df_mx)

# Configurar el estilo de Seaborn y la paleta de colores
sns.set_theme(style='whitegrid')

# Crear una paleta de colores de negro a azul
unique_cl = keyword_counts_cl['criterio_clasificacion'].nunique()
unique_mx = keyword_counts_mx['criterio_clasificacion'].nunique()
max_unique = max(unique_cl, unique_mx)
palette = sns.dark_palette("blue", reverse=False, n_colors=max_unique)

# Crear la grilla de gráficos con Seaborn
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Crear el gráfico de Chile
sns.barplot(ax=axes[0], x='counts', y='criterio_clasificacion', hue='criterio_clasificacion', data=keyword_counts_cl, palette=palette, dodge=False, legend=False)
axes[0].set_title('Chile', fontsize=22)
axes[0].set_xlabel('')
axes[0].set_ylabel('Tipo de Keyword')
for i in range(len(keyword_counts_cl)):
    axes[0].text(keyword_counts_cl['counts'][i] + 0.5, i, f'{keyword_counts_cl["counts"][i]} ({keyword_counts_cl["percentage"][i]:.2f}%)', va='center')

# Crear el gráfico de México
sns.barplot(ax=axes[1], x='counts', y='criterio_clasificacion', hue='criterio_clasificacion', data=keyword_counts_mx, palette=palette, dodge=False, legend=False)
axes[1].set_title('México', fontsize=22)
axes[1].set_xlabel('Cantidad')
axes[1].set_ylabel('Tipo de Keyword')
for i in range(len(keyword_counts_mx)):
    axes[1].text(keyword_counts_mx['counts'][i] + 0.5, i, f'{keyword_counts_mx["counts"][i]} ({keyword_counts_mx["percentage"][i]:.2f}%)', va='center')

# Ajustar los márgenes y añadir un título global
plt.subplots_adjust(top=0.92, hspace=0.3)
fig.suptitle('Cantidad y Porcentaje de Registros por Tipo de Keyword por País', fontsize=18)

# Guardar los gráficos como un archivo PNG con fondo transparente
plt.savefig('grafico_barras_tipo_keyword_adjusted_vertical.png', transparent=True)

# Guardar los gráficos como un archivo PNG con fondo blanco
plt.savefig('grafico_barras_tipo_keyword_adjusted_vertical_not_transparent.png')

# Mostrar los gráficos
plt.show()

print("El gráfico de barras con la cantidad y porcentaje de tipos de keyword ha sido guardado como 'grafico_barras_tipo_keyword_adjusted_vertical.png'")

In [ ]:
# Convertir el DataFrame de polars a pandas
df_pandas = df.to_pandas()

# df2 a pandas
df2_pandas = df2.to_pandas()

In [ ]:
# Función para expandir categorías en columnas
def expand_categories(df):
    print("Iniciando la expansión de categorías...")
    print(f"Altura original del DataFrame: {df.shape[0]}")
    print(f"Columnas originales del DataFrame: {df.columns}")

    # Obtener el número máximo de categorías
    max_categories = df['categorias'].apply(len).max()
    print(f"Número máximo de categorías: {max_categories}")

    # Crear nuevas columnas para las categorías usando .loc para evitar SettingWithCopyWarning
    for i in range(max_categories):
        df.loc[:, f'categoria_{i + 1}'] = df['categorias'].apply(lambda x: x[i] if i < len(x) else None)

    print(f"Columnas después de la expansión: {df.columns}")
    print(f"Altura del DataFrame después de la expansión: {df.shape[0]}")

    return df

# Filtrar el DataFrame original para obtener filas con al menos dos categorías
df_filtrado = df_pandas[df_pandas['cantidad_categorias'] >= 2].copy()
print(f"DataFrame filtrado para al menos dos categorías, altura: {df_filtrado.shape[0]}")

# Aplicar la función al DataFrame filtrado
expanded_df = expand_categories(df_filtrado)

# Seleccionar las columnas relevantes que existen en el DataFrame
relevant_columns = ['cluster_index', 'country'] + [col for col in expanded_df.columns if col.startswith('categoria_')]
expanded_df = expanded_df[relevant_columns].drop_duplicates()
print("Después de eliminar duplicados:")
print(expanded_df)

# drop categoria_principal & rename categoria_1 to categoria_principal
expanded_df = expanded_df.drop('categoria_1', axis=1)

# Filtrar filas donde categoria_2 no es nulo
expanded_df = expanded_df[expanded_df['categoria_2'].notnull()]
print(f"Después de filtrar donde 'categoria_2' no es nulo, altura: {expanded_df.shape[0]}")

# Identificar dinámicamente las columnas de categorías para ordenar
category_columns_to_sort = [col for col in expanded_df.columns if col.startswith('categoria_')]
print(f"Columnas de categoría para ordenar: {category_columns_to_sort}")

# Ordenar las filas alfabéticamente por los valores de las columnas de categorías (de derecha a izquierda)
expanded_df = expanded_df.sort_values(by=category_columns_to_sort[::-1] + ['cluster_index'])
print("Después de ordenar:")
print(expanded_df)

# Resetear el índice del DataFrame
expanded_df = expanded_df.reset_index(drop=True)

# Mostrar el DataFrame final
print("DataFrame final:")
print(expanded_df)

# Mostrar un mensaje de advertencia si el DataFrame está vacío
if expanded_df.empty:
    print("Advertencia: No hay registros disponibles después de la expansión y filtrado.")
else:
    display(expanded_df)

In [ ]:
# Contar las ocurrencias de cada cluster_index por categoría principal y por país
categoria_counts = df_pandas.groupby(['categoria_principal', 'country'])['cluster_index'].nunique().reset_index()
categoria_counts.columns = ['categoria_principal', 'country', 'count']

# Ordenar por cantidad de mayor a menor
categoria_counts = categoria_counts.sort_values(by=['categoria_principal', 'country'])

# Restablecer el índice
categoria_counts = categoria_counts.reset_index(drop=True)

# Mostrar el DataFrame
categoria_counts

In [ ]:
# Filtrar el DataFrame original para obtener filas con cantidad_categorias == 1
df_filtrado = df_pandas[df_pandas['cantidad_categorias'] == 1].copy()

# Crear la columna 'other_categories' con valores nulos
df_filtrado['otras_categorias'] = pd.NA

# Crear la columna 'load_date' con la fecha y hora actual
df_filtrado['load_date'] = datetime.now()

# Seleccionar las columnas en el orden requerido
df_final = df_filtrado[['cluster_index', 'categoria_principal', 'otras_categorias', 'criterio_clasificacion', 'tipo_clasificacion', 'country', 'load_date']]

# Renombrar las columnas al inglés
df_final.columns = ['cluster_index', 'main_category', 'other_categories', 'classification_criteria', 'classification_type', 'country', 'load_date']

# Resetear el índice
df_final = df_final.reset_index(drop=True)

# Mostrar el DataFrame final
df_final

In [ ]:
# Esquema a utilizar
schema_name = "esquema_analitico"

# Función para crear la tabla si no existe en el esquema especificado
def create_table_if_not_exists(conn, table_name, schema):
    full_table_name = f"{schema_name}.{table_name}"
    print(f"\nCreating table {full_table_name} if not exists...\n")
    with conn.cursor() as cursor:
        try:
            cursor.execute(f"""
            CREATE TABLE IF NOT EXISTS {full_table_name} (
                {schema}
            )
            """)
            conn.commit()
            print(f"Table {full_table_name} created or already exists.\n")
        except Exception as e:
            print(f"Error creating table {full_table_name}: {e}\n")

# Función para obtener los datos actuales de la tabla
def get_existing_data(conn, table_name, primary_key):
    print(f"\nGetting existing data for table {schema_name}.{table_name}...\n")
    full_table_name = f"{schema_name}.{table_name}"
    with conn.cursor(name='fetch_cursor') as cursor:  # Use server-side cursor to handle large datasets
        cursor.execute(f"SELECT {primary_key} FROM {full_table_name}")
        existing_keys = set(row[0] for row in cursor)
        print(f"Retrieved {len(existing_keys)} existing keys from {full_table_name}.\n")
        return existing_keys

# Función para verificar si la tabla ya contiene datos
def table_has_data(conn, table_name):
    full_table_name = f"{schema_name}.{table_name}"
    print(f"\nChecking if table {full_table_name} has data...\n")
    with conn.cursor() as cursor:
        cursor.execute(f"SELECT COUNT(*) FROM {full_table_name}")
        count = cursor.fetchone()[0]
        print(f"Table {full_table_name} has {count} rows.\n")
        return count > 0

# Función para verificar y actualizar o insertar datos en la tabla
def upsert_data(conn, table_name, data, primary_key, batch_size=1000):
    if isinstance(data, pd.DataFrame) and data.empty:
        print(f"No data to upsert for table {table_name}.\n")
        return

    full_table_name = f"{schema_name}.{table_name}"
    columns = ', '.join(data.columns)
    placeholders = ', '.join(['%s'] * len(data.columns))
    update_placeholders = ', '.join([f"{col} = EXCLUDED.{col}" for col in data.columns])
    existing_keys = get_existing_data(conn, table_name, primary_key)

    duplicate_count = 0
    update_count = 0
    insert_count = 0
    print(f"\nUpserting data into table {full_table_name}...\n")
    
    # Process in batches
    total_rows = len(data)
    for start in range(0, total_rows, batch_size):
        end = min(start + batch_size, total_rows)
        batch_data = data[start:end]
        with conn.cursor() as cursor:
            try:
                for i, row in batch_data.iterrows():
                    # Convertir valores de tipo diccionario a JSON y numpy arrays a listas
                    row_dict = row.to_dict()
                    row_dict = {k: json.dumps(v) if isinstance(v, dict) else (v.tolist() if isinstance(v, (np.ndarray, list)) else v) for k, v in row_dict.items()}
                    pk_value = row_dict[primary_key]
                    if pk_value in existing_keys:
                        # Comparar columnas
                        identical = True  # Assume identical unless any column differs
                        if identical:
                            duplicate_count += 1
                        else:
                            cursor.execute(f"""
                            INSERT INTO {full_table_name} ({columns})
                            VALUES ({placeholders})
                            ON CONFLICT ({primary_key}) DO UPDATE
                            SET {update_placeholders}
                            """, list(row_dict.values()))
                            update_count += 1
                    else:
                        cursor.execute(f"""
                        INSERT INTO {full_table_name} ({columns})
                        VALUES ({placeholders})
                        ON CONFLICT ({primary_key}) DO UPDATE
                        SET {update_placeholders}
                        """, list(row_dict.values()))
                        insert_count += 1
                conn.commit()
                print(f"Processed rows {start+1}-{end} of {total_rows}.\n")
            except Exception as e:
                print(f"Error upserting data into table {full_table_name}: {e}\n")

    print(f"Data upserted into table {full_table_name}.\n")
    print(f"Number of duplicate rows found: {duplicate_count}\n")
    print(f"Number of rows updated: {update_count}\n")
    print(f"Number of rows inserted: {insert_count}\n")

# Creación de las tablas y exportación a Excel
def process_and_create_tables(df_final, table_name, excel_path, primary_key):
    print(f"\nProcessing and creating table {table_name}...\n")
    # Crear esquema dinámico basado en las columnas del DataFrame
    dtype_mapping = {pd.StringDtype(): 'VARCHAR', pd.Int64Dtype(): 'INT', pd.Float64Dtype(): 'FLOAT'}
    schema = ",\n".join([f"{col} {dtype_mapping.get(dtype, 'VARCHAR')}" for col, dtype in zip(df_final.columns, df_final.dtypes)])
    schema = f"{schema},\nPRIMARY KEY ({primary_key})"

    print(f"Schema for table {table_name}:\n{schema}\n")
    create_table_if_not_exists(conn, table_name, schema)

    # Verificar si la tabla ya contiene datos
    if table_has_data(conn, table_name):
        print(f"Table {schema_name}.{table_name} already contains data.\n")
    else:
        print(f"Table {schema_name}.{table_name} is empty.\n")

    # Insertar o actualizar datos en la tabla
    upsert_data(conn, table_name, df_final, primary_key)

    # Exportar a Excel
    print(f"Exporting data to Excel {excel_path}...\n")
    df_final.to_excel(excel_path, index=False)

    # Ajustar el archivo Excel
    wb = load_workbook(excel_path)
    ws = wb.active
    ws.auto_filter.ref = ws.dimensions
    ws.freeze_panes = 'A2'

    for column in ws.columns:
        max_length = 0
        column = list(column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(cell.value)
            except:
                pass
        adjusted_width = (max_length + 2)
        ws.column_dimensions[get_column_letter(column[0].column)].width = adjusted_width

    wb.save(excel_path)
    print(f"Datos exportados a Excel en '{excel_path}'.\n")

# Ejemplo de uso con la tabla `df_final`
process_and_create_tables(df_final, 'ctrl_categorias', 'ctrl_categorias.xlsx', 'cluster_index')

# Ejemplo de uso con la tabla `df2_pandas`
process_and_create_tables(df2_pandas, 'ctrl_categorias_keywords', 'ctrl_categorias_keywords.xlsx', 'keyword')

In [ ]:
# Cerrar la conexión a la base de datos
conn.close()
print("Conexión a la base de datos cerrada.")

In [ ]:
# Cargar el archivo notebook
notebook_path = 'nombres_polars.ipynb'
with open(notebook_path, 'r', encoding='utf-8') as f:
    notebook = nbformat.read(f, as_version=4)

# Extraer y concatenar las celdas de código
code_cells = [cell['source'] for cell in notebook.cells if cell.cell_type == 'code']
complete_code = "\n\n".join(code_cells)

# Guardar el código en un archivo de texto
output_path = 'codigo_notebook.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(complete_code)

print(f"Todo el código del notebook ha sido exportado a {output_path}")